# Problem 2 – Transformer from Scratch
**Student:** Ayushi Pradhan &nbsp;|&nbsp; **ID:** 2026AJ05077  
**d=128**, **4 heads**, **4 layers**, context=128  
**Seeds:** 0, 1, 2


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports, seed helper and device                       ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, random, time, math, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Hyper-parameters (Problem 2 architecture) ─────────────────────────────────
D_MODEL   = 128    # embedding / model dimension
N_HEADS   = 4      # number of attention heads
N_LAYERS  = 4      # encoder depth
D_FF      = 512    # feed-forward inner dimension (4 × D_MODEL)
DROPOUT   = 0.1
CONTEXT   = 128    # context window for language model
SEEDS     = [0, 1, 2]

FIG_DIR   = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
print("Setup complete.")


Device: cuda
Setup complete.


## Task 1 – Attention from Scratch

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Scaled dot-product attention from scratch             ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Formula:
#   Attention(Q, K, V) = softmax( (Q K^T) / √d_k  [+ mask] ) V
#
# Parameters
# ----------
# Q, K, V  : (B, n_heads, T, d_k)  or  (B, T, d_k) for single-head
# mask     : (T, T) boolean or None  — True means MASK OUT (set to -inf)
#
# The mask is used for causal (autoregressive) language modelling.

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention.

    Q : (..., T_q, d_k)
    K : (..., T_k, d_k)
    V : (..., T_k, d_v)
    mask : (..., T_q, T_k) bool – positions to MASK OUT (future positions)

    Returns
    -------
    output : (..., T_q, d_v)
    weights : (..., T_q, T_k)
    """
    d_k = Q.size(-1)                               # key dimension

    # Raw attention scores: (B, heads, T_q, T_k)
    scores = torch.matmul(Q, K.transpose(-2, -1))  # Q K^T
    scores = scores / math.sqrt(d_k)              # scale by 1/√d_k

    if mask is not None:
        # Positions where mask is True are set to -∞ so softmax → 0
        scores = scores.masked_fill(mask, float('-inf'))

    weights = F.softmax(scores, dim=-1)            # (…, T_q, T_k)
    output  = torch.matmul(weights, V)             # (…, T_q, d_v)
    return output, weights


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Attention unit tests                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

set_seed(0)
B, T, d_k = 4, 8, 32

Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_k)

# Test 1: attention weights sum to 1 along the key dimension
out, w = scaled_dot_product_attention(Q, K, V)
row_sums = w.sum(dim=-1)   # (B, T)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), \
    "FAIL: attention row weights do not sum to 1"
print("✓ Test 1: attention weights sum to 1 (atol=1e-5)")

# Test 2: matches torch.nn.functional.scaled_dot_product_attention (no mask)
ref_out = F.scaled_dot_product_attention(Q, K, V)
max_err = (out - ref_out).abs().max().item()
assert max_err < 1e-5, f"FAIL: output differs from reference by {max_err}"
print(f"✓ Test 2: matches F.scaled_dot_product_attention within 1e-5 (max diff={max_err:.2e})")

# Test 3: causal mask → zero weight to future positions
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
# shape (T, T): True = mask out upper triangle (future positions)
out_masked, w_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
# Upper triangle of w should be 0
upper = w_masked[:, torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)]
assert upper.abs().max().item() < 1e-6, "FAIL: future positions not masked"
print("✓ Test 3: causal mask gives zero weight to future positions")

# ── Hand-checkable 3-token example ───────────────────────────────────────────
# Instructor-supplied example: d_k=2 for easy manual verification
Q3 = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]])      # (1,3,2)
K3 = torch.tensor([[[1., 0.], [0., 1.], [0., 0.]]])
V3 = torch.tensor([[[1., 0.], [0., 1.], [0.5, 0.5]]])

out3, w3 = scaled_dot_product_attention(Q3, K3, V3)
print("\n── 3-token hand example ─────────────────────────────")
print(f"Q:\n{Q3[0].numpy()}\nK:\n{K3[0].numpy()}\nV:\n{V3[0].numpy()}")
print(f"Scores (Q K^T / √2):\n{(Q3 @ K3.transpose(-2,-1) / math.sqrt(2))[0].detach().numpy()}")
print(f"Weights:\n{w3[0].detach().numpy().round(4)}")
print(f"Output:\n{out3[0].detach().numpy().round(4)}")
print("(Verify row 0 manually: scores = [1/√2, 0, 0] → softmax → weights → weighted V)")


✓ Test 1: attention weights sum to 1 (atol=1e-5)
✓ Test 2: matches F.scaled_dot_product_attention within 1e-5 (max diff=2.38e-07)
✓ Test 3: causal mask gives zero weight to future positions

── 3-token hand example ─────────────────────────────
Q:
[[1. 0.]
 [0. 1.]
 [1. 1.]]
K:
[[1. 0.]
 [0. 1.]
 [0. 0.]]
V:
[[1.  0. ]
 [0.  1. ]
 [0.5 0.5]]
Scores (Q K^T / √2):
[[0.70710677 0.         0.        ]
 [0.         0.70710677 0.        ]
 [0.70710677 0.70710677 0.        ]]
Weights:
[[0.5035 0.2483 0.2483]
 [0.2483 0.5035 0.2483]
 [0.4011 0.4011 0.1978]]
Output:
[[0.6276 0.3724]
 [0.3724 0.6276]
 [0.5    0.5   ]]
(Verify row 0 manually: scores = [1/√2, 0, 0] → softmax → weights → weighted V)


## Task 2 – Multi-Head Attention and Encoder Block

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Multi-head attention (W_Q, W_K, W_V, W_O)            ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Multi-head attention:
#   head_i  = Attention( Q W_Q_i, K W_K_i, V W_V_i )
#   MHA     = Concat(head_1, …, head_h) W_O
#
# We store W_Q, W_K, W_V as single (d, d) matrices, then split
# into n_heads slices of d_k = d / n_heads each.

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention built from scaled dot-product attention.

    Parameters
    ----------
    d_model : model dimension (must be divisible by n_heads)
    n_heads : number of parallel attention heads
    dropout : dropout on attention weights
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads    # dimension per head

        # Projection matrices — stored as (d, d) for efficiency
        # W_Q: (d, d),  W_K: (d, d),  W_V: (d, d),  W_O: (d, d)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

        # Store last attention weights for visualisation (Task 5a)
        self.last_weights = None

    def forward(self, Q, K, V, mask=None):
        """Multi-head attention forward pass.

        Q, K, V : (B, T, d_model)
        mask    : (T, T) bool or None

        Returns : (B, T, d_model)
        """
        B, T, _ = Q.shape

        # Project and reshape to (B, n_heads, T, d_k)
        # 1. Linear project: (B, T, d_model) → (B, T, d_model)
        # 2. Reshape:        (B, T, d_model) → (B, T, n_heads, d_k)
        # 3. Transpose:      (B, T, n_heads, d_k) → (B, n_heads, T, d_k)
        def project_and_split(W, x):
            return W(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        Q_h = project_and_split(self.W_Q, Q)   # (B, n_heads, T, d_k)
        K_h = project_and_split(self.W_K, K)
        V_h = project_and_split(self.W_V, V)

        # Expand mask to (1, 1, T, T) for broadcasting across B and heads
        mask_4d = mask.unsqueeze(0).unsqueeze(0) if mask is not None else None

        # Scaled dot-product attention for all heads simultaneously
        attn_out, attn_w = scaled_dot_product_attention(Q_h, K_h, V_h, mask_4d)
        # attn_out : (B, n_heads, T, d_k)

        # Apply dropout to attention weights
        attn_w = self.attn_dropout(attn_w)
        self.last_weights = attn_w.detach()   # save for visualisation

        # Recombine heads: (B, n_heads, T, d_k) → (B, T, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.d_model)

        # Final output projection W_O
        return self.W_O(attn_out)   # (B, T, d_model)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Sinusoidal positional encoding                        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# PE(pos, 2i)   = sin( pos / 10000^{2i/d} )
# PE(pos, 2i+1) = cos( pos / 10000^{2i/d} )
#
# This encoding is fixed (not learned), added to the token embeddings.

class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding.

    Parameters
    ----------
    d_model  : embedding dimension
    max_len  : maximum sequence length supported
    dropout  : dropout applied after adding PE to embeddings
    """
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build PE table: (max_len, d_model)
        pe  = torch.zeros(max_len, d_model)                      # (L, d)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)  # (L, 1)
        # Frequencies: 1 / 10000^{2i/d}
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) *
            (-math.log(10000.0) / d_model)
        )   # (d/2,)
        pe[:, 0::2] = torch.sin(pos * div)   # even indices → sin
        pe[:, 1::2] = torch.cos(pos * div)   # odd  indices → cos

        # Register as buffer (not a parameter, moves with model.to(device))
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, L, d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input embeddings.

        x : (B, T, d_model)
        """
        x = x + self.pe[:, :x.size(1), :]   # broadcast over batch
        return self.dropout(x)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Pre-LayerNorm encoder block                           ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Pre-LN (apply LayerNorm BEFORE sublayer):
#   x = x + MHA( LN(x) )
#   x = x + FFN( LN(x) )
#
# Feed-forward: Linear(d, d_ff) → ReLU → Linear(d_ff, d)   (d_ff = 4d)
#
# Weight count per block:
#   MHA   : W_Q + W_K + W_V + W_O = 4 × d²   (no biases in our W_O/W_Q etc.)
#   FFN   : d×d_ff (+ bias d_ff) + d_ff×d (+ bias d) = 2 × d × d_ff + d_ff + d
#   LN×2  : 2 × (2d) parameters (weight + bias)
#   Total ≈ 4d² + 2×d×d_ff  (dominant terms)
# For d=128, d_ff=512: 4×128²+2×128×512 = 65,536+131,072 = 196,608 (weights only)

class EncoderBlock(nn.Module):
    """Single Transformer encoder block (pre-LayerNorm).

    Parameters
    ----------
    d_model  : embedding dimension
    n_heads  : number of attention heads
    d_ff     : inner dimension of the FFN (usually 4 × d_model)
    dropout  : dropout rate
    """
    def __init__(self, d_model: int, n_heads: int,
                 d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn      = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn       = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        # Pre-layer normalisation
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                mask=None) -> torch.Tensor:
        """Pre-LN encoder block forward.

        x    : (B, T, d_model)
        mask : (T, T) bool or None (causal mask for LM)

        Returns : (B, T, d_model)
        """
        # Sub-layer 1: multi-head self-attention with residual connection
        # LN → MHA → dropout → residual
        x_norm = self.norm1(x)
        x = x + self.dropout(self.attn(x_norm, x_norm, x_norm, mask=mask))

        # Sub-layer 2: position-wise FFN with residual connection
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ffn(x_norm))
        return x


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Parameter-count verification                          ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Derive theoretically and reconcile with nn.Module.
#
# For d=128, d_ff=512, n_heads=4, d_k=32:
#   MHA weights only (no biases): 4 × d² = 4 × 128² = 65,536
#   FFN  weights + biases:
#       Linear(128,512) : 128×512 weights + 512 bias  = 66,048
#       Linear(512,128) : 512×128 weights + 128 bias  = 65,664
#       Total FFN                                      = 131,712
#   LN × 2             : 2 × 2 × 128                 = 512
#   ─────────────────────────────────────────────────────────────
#   Per-block total                                   = 197,760
#   (Weights-only formula 4d² + 2×d×d_ff = 196,608  → biases + LN add 1,152)

set_seed(0)
block = EncoderBlock(D_MODEL, N_HEADS, D_FF, DROPOUT)
total = sum(p.numel() for p in block.parameters())
print(f"EncoderBlock (d={D_MODEL}, heads={N_HEADS}, d_ff={D_FF})")
print(f"  Measured parameter count : {total:,}")

# Break down by sub-module
for name, module in block.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params:
        print(f"  {name:30s}: {params:,}")

# Theoretical (weights only)
theory_weights = 4 * D_MODEL**2 + 2 * D_MODEL * D_FF
print(f"\nTheoretical (weights only 4d²+2d·d_ff): {theory_weights:,}")
print(f"Difference (biases + LN):                {total - theory_weights:,}")

# Verify shapes
dummy_in = torch.randn(2, 16, D_MODEL)   # (B=2, T=16, d=128)
dummy_out = block(dummy_in)
assert dummy_out.shape == dummy_in.shape, "Shape mismatch in encoder block"
print(f"\n✓ Encoder block output shape: {tuple(dummy_out.shape)}  (matches input)")


EncoderBlock (d=128, heads=4, d_ff=512)
  Measured parameter count : 197,760
  attn.W_Q                      : 16,384
  attn.W_K                      : 16,384
  attn.W_V                      : 16,384
  attn.W_O                      : 16,384
  ffn.0                         : 66,048
  ffn.2                         : 65,664
  norm1                         : 256
  norm2                         : 256

Theoretical (weights only 4d²+2d·d_ff): 196,608
Difference (biases + LN):                1,152

✓ Encoder block output shape: (2, 16, 128)  (matches input)


## Task A – Encoder Classifier on HAR

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Load and normalise UCI HAR (reuse from Problem 1)     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# We re-load the dataset here so Problem 2 is self-contained.
# (In Colab, Problem 1 is typically run first so the files are cached.)

import zipfile as zf, os, urllib.request

HAR_URL   = ('https://archive.ics.uci.edu/static/public/240/'
             'human+activity+recognition+using+smartphones.zip')
HAR_ZIP   = 'har.zip'
HAR_DIR   = 'har'

if not os.path.isdir(os.path.join(HAR_DIR, 'UCI HAR Dataset')):
    print("Downloading UCI HAR…")
    urllib.request.urlretrieve(HAR_URL, HAR_ZIP)
    with zf.ZipFile(HAR_ZIP) as z:
        z.extractall(HAR_DIR)
    inner = os.path.join(HAR_DIR, 'UCI HAR Dataset.zip')
    if os.path.isfile(inner):
        with zf.ZipFile(inner) as z:
            z.extractall(HAR_DIR)
HAR_ROOT = os.path.join(HAR_DIR, 'UCI HAR Dataset')

CHANNELS = ['body_acc_x','body_acc_y','body_acc_z',
            'body_gyro_x','body_gyro_y','body_gyro_z',
            'total_acc_x','total_acc_y','total_acc_z']
ACTIVITY_LABELS = ['Walking','Walk Up','Walk Down','Sitting','Standing','Laying']

def load_har_split(split):
    sig_dir = os.path.join(HAR_ROOT, split, 'Inertial Signals')
    channels = [np.loadtxt(os.path.join(sig_dir, f'{ch}_{split}.txt'),
                           dtype=np.float32) for ch in CHANNELS]
    X        = np.stack(channels, axis=-1)          # (N,128,9)
    y        = np.loadtxt(os.path.join(HAR_ROOT, split, f'y_{split}.txt'),
                          dtype=np.int64) - 1
    subjects = np.loadtxt(os.path.join(HAR_ROOT, split, f'subject_{split}.txt'),
                          dtype=np.int64)
    return X, y, subjects

X_train_full, y_train_full, subj_train = load_har_split('train')
X_test,       y_test,       _          = load_har_split('test')

def make_val_split(X_full, y_full, subjects_full, seed):
    rng  = np.random.default_rng(seed)
    unique_subjs = np.unique(subjects_full)
    held = rng.choice(unique_subjs, 4, replace=False)
    mask = np.isin(subjects_full, held)
    X_tr, y_tr   = X_full[~mask], y_full[~mask]
    X_v,  y_v    = X_full[mask],  y_full[mask]
    mean = X_tr.mean(axis=(0,1), keepdims=True)
    std  = X_tr.std( axis=(0,1), keepdims=True) + 1e-8
    return (X_tr-mean)/std, y_tr, (X_v-mean)/std, y_v, mean, std

X_tr, y_tr, X_val, y_val, har_mean, har_std = make_val_split(
    X_train_full, y_train_full, subj_train, seed=0)
X_test_norm = (X_test - har_mean) / har_std
print(f"HAR ready: train={X_tr.shape}, val={X_val.shape}, test={X_test_norm.shape}")


HAR ready: train=(5886, 128, 9), val=(1466, 128, 9), test=(2947, 128, 9)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Transformer HAR classifier with CLS token             ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Architecture:
#   Input: (B, 128, 9) sensor windows
#   1. Patch embedding: each time-step projected from 9 → d_model=128
#      (or patch of 8 steps: 128/8=16 patches of 8×9=72 features → d)
#   2. Prepend CLS token (learnable, (1, 1, d))
#   3. Add positional encoding (sinusoidal or learned, selectable)
#   4. N encoder blocks
#   5. CLS token output → linear head → 6 classes

class TransformerHARClassifier(nn.Module):
    """Transformer encoder classifier for UCI HAR.

    Parameters
    ----------
    input_channels : 9  (sensor channels)
    seq_len        : 128 (time steps)
    d_model        : model dimension
    n_heads        : attention heads
    n_layers       : encoder depth
    n_classes      : 6
    pos_type       : 'sinusoidal' | 'learned' | 'none'
    """
    def __init__(self,
                 input_channels: int = 9,
                 seq_len: int = 128,
                 d_model: int = D_MODEL,
                 n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS,
                 n_classes: int = 6,
                 pos_type: str = 'sinusoidal',
                 dropout: float = DROPOUT):
        super().__init__()
        self.d_model  = d_model
        self.pos_type = pos_type

        # Step embedding: project each 9-dim time step to d_model
        self.input_proj = nn.Linear(input_channels, d_model)

        # CLS token: learnable (1, 1, d)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, d_model))

        # Positional encoding for seq_len + 1 (extra for CLS)
        if pos_type == 'sinusoidal':
            self.pos_enc = SinusoidalPositionalEncoding(d_model,
                                                        max_len=seq_len + 1,
                                                        dropout=dropout)
        elif pos_type == 'learned':
            self.pos_enc = nn.Embedding(seq_len + 1, d_model)
            self.pos_ids = torch.arange(seq_len + 1)   # (T+1,) positions
        # 'none': no positional encoding

        # Encoder stack
        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_model * 4, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, 128, 9) → logits (B, 6)"""
        B = x.size(0)

        # Project sensor channels to d_model: (B, T, d_model)
        x = self.input_proj(x)

        # Prepend CLS token: (B, T+1, d_model)
        cls = self.cls_token.expand(B, -1, -1)   # (B, 1, d)
        x   = torch.cat([cls, x], dim=1)          # (B, T+1, d)

        # Positional encoding
        if self.pos_type == 'sinusoidal':
            x = self.pos_enc(x)
        elif self.pos_type == 'learned':
            ids = self.pos_ids.to(x.device)
            x   = x + self.pos_enc(ids).unsqueeze(0)
        # 'none': x unchanged

        # Encoder blocks (no mask for classification)
        for block in self.encoder:
            x = block(x)

        x = self.norm(x)
        # Classify using the CLS token (position 0)
        cls_out = x[:, 0, :]      # (B, d)
        return self.head(self.dropout(cls_out))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick shape check
for pt in ['sinusoidal', 'learned', 'none']:
    m = TransformerHARClassifier(pos_type=pt).to(DEVICE)
    out = m(torch.randn(4, 128, 9).to(DEVICE))
    assert out.shape == (4, 6)
    print(f"pos_type={pt:12s} | params={count_params(m):,} | output={tuple(out.shape)}")


pos_type=sinusoidal   | params=793,478 | output=(4, 6)
pos_type=learned      | params=809,990 | output=(4, 6)
pos_type=none         | params=793,478 | output=(4, 6)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Train HAR Transformer and positional-encoding ablation║
# ╚══════════════════════════════════════════════════════════════════╝

N_EPOCHS_HAR2 = 25
BATCH_SIZE    = 128

def train_har_transformer(pos_type, seed,
                          n_epochs=N_EPOCHS_HAR2):
    set_seed(seed)
    X_tr2, y_tr2, X_v2, y_v2, m2, s2 = make_val_split(
        X_train_full, y_train_full, subj_train, seed)
    X_te2 = (X_test - m2) / s2

    def make_loader(X, y, sh):
        ds = TensorDataset(torch.tensor(X),
                           torch.tensor(y, dtype=torch.long))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh)

    tl = make_loader(X_tr2, y_tr2, True)
    vl = make_loader(X_v2,  y_v2,  False)
    tel= make_loader(X_te2, y_test, False)

    model = TransformerHARClassifier(pos_type=pos_type).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit  = nn.CrossEntropyLoss()

    best_val, best_state = float('inf'), None
    t_losses, v_losses   = [], []
    total_t = 0.0

    for ep in range(n_epochs):
        model.train()
        t0 = time.time()
        for Xb, yb in tl:
            opt.zero_grad()
            crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        total_t += time.time() - t0

        model.eval()
        with torch.no_grad():
            vl_epoch = np.mean([crit(model(Xb.to(DEVICE)),
                                     yb.to(DEVICE)).item() for Xb,yb in vl])
            tl_epoch = np.mean([crit(model(Xb.to(DEVICE)),
                                     yb.to(DEVICE)).item() for Xb,yb in tl])
        t_losses.append(tl_epoch); v_losses.append(vl_epoch)
        if vl_epoch < best_val:
            best_val   = vl_epoch
            best_state = {k: v.clone() for k,v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for Xb, yb in tel:
            all_p.extend(model(Xb.to(DEVICE)).argmax(1).cpu().numpy())
            all_t.extend(yb.numpy())

    return {
        'test_acc':      accuracy_score(all_t, all_p),
        'macro_f1':      f1_score(all_t, all_p, average='macro'),
        'params':        count_params(model),
        'sec_per_epoch': total_t / n_epochs,
        't_losses':      t_losses,
        'v_losses':      v_losses,
        'model':         model,
    }

print("Training Transformer HAR classifiers (3 pos_type × 1 seed)…")
pos_results = {}
for pt in ['sinusoidal', 'learned', 'none']:
    print(f"  pos_type={pt} …", end='', flush=True)
    pos_results[pt] = train_har_transformer(pt, seed=0)
    print(f" acc={pos_results[pt]['test_acc']:.4f}")

# Multi-seed for sinusoidal (fair comparison with Problem 1)
print("Multi-seed run for sinusoidal (3 seeds)…")
sin_multi = []
for seed in SEEDS:
    print(f"  seed={seed} …", end='', flush=True)
    r = train_har_transformer('sinusoidal', seed)
    sin_multi.append(r)
    print(f" acc={r['test_acc']:.4f}")


Training Transformer HAR classifiers (3 pos_type × 1 seed)…
  pos_type=sinusoidal … acc=0.9019
  pos_type=learned … acc=0.8989
  pos_type=none … acc=0.9046
Multi-seed run for sinusoidal (3 seeds)…
  seed=0 … acc=0.9019
  seed=1 … acc=0.8873
  seed=2 … acc=0.8690


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Table 3 & Figure 8: Task A results                   ║
# ╚══════════════════════════════════════════════════════════════════╝

# Table 3: Transformer vs best P1 model on HAR
accs_sin = [r['test_acc'] for r in sin_multi]
f1s_sin  = [r['macro_f1'] for r in sin_multi]
print("Table 3: HAR comparison – Transformer vs best Problem 1 model (LSTM/GRU)")
p2_rows = []
p2_rows.append({
    'Model': 'Transformer (sinusoidal)',
    'Params': sin_multi[0]['params'],
    'Acc (mean±std)': f"{np.mean(accs_sin):.4f}±{np.std(accs_sin):.4f}",
    'F1 (mean±std)':  f"{np.mean(f1s_sin):.4f}±{np.std(f1s_sin):.4f}",
    'Sec/ep': f"{np.mean([r['sec_per_epoch'] for r in sin_multi]):.2f}",
})
# Positional encoding ablation (seed 0)
for pt in ['sinusoidal', 'learned', 'none']:
    r = pos_results[pt]
    p2_rows.append({
        'Model': f'Transformer ({pt})',
        'Params': r['params'],
        'Acc (mean±std)': f"{r['test_acc']:.4f} (seed 0)",
        'F1 (mean±std)':  f"{r['macro_f1']:.4f} (seed 0)",
        'Sec/ep': f"{r['sec_per_epoch']:.2f}",
    })

tbl3 = pd.DataFrame(p2_rows)
print(tbl3.to_string(index=False))

# Figure 8: learning curves (pos encoding ablation)
fig, ax = plt.subplots(figsize=(8, 4))
color_map = {'sinusoidal':'tab:blue','learned':'tab:orange','none':'tab:red'}
for pt in ['sinusoidal','learned','none']:
    r = pos_results[pt]
    ep = range(1, len(r['v_losses'])+1)
    ax.plot(ep, r['t_losses'], color=color_map[pt], alpha=0.4, lw=1)
    ax.plot(ep, r['v_losses'], color=color_map[pt], lw=2, ls='--', label=pt)
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('Figure 8: Task A – positional encoding ablation (HAR)')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/'fig8_taskA_pos_ablation.png', dpi=150, bbox_inches='tight')
plt.show(); print("Figure 8 saved.")


Table 3: HAR comparison – Transformer vs best Problem 1 model (LSTM/GRU)
                   Model  Params  Acc (mean±std)   F1 (mean±std) Sec/ep
Transformer (sinusoidal)  793478   0.8861±0.0135   0.8842±0.0146   2.99
Transformer (sinusoidal)  793478 0.9019 (seed 0) 0.9009 (seed 0)   2.85
   Transformer (learned)  809990 0.8989 (seed 0) 0.8987 (seed 0)   2.93
      Transformer (none)  793478 0.9046 (seed 0) 0.9024 (seed 0)   2.94
Figure 8 saved.


## Task B – Character-Level Causal Language Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Tiny Shakespeare download and vocabulary             ║
# ╚══════════════════════════════════════════════════════════════════╝

SHAKE_URL = ('https://raw.githubusercontent.com/karpathy/char-rnn/'
             'master/data/tinyshakespeare/input.txt')
SHAKE_FILE = 'tinyshakespeare.txt'

if not os.path.isfile(SHAKE_FILE):
    print("Downloading Tiny Shakespeare…")
    text = urllib.request.urlopen(SHAKE_URL).read().decode('utf-8')
    with open(SHAKE_FILE, 'w') as f:
        f.write(text)
else:
    with open(SHAKE_FILE) as f:
        text = f.read()

print(f"Total characters: {len(text):,}")

# Build character vocabulary from the WHOLE text (65 characters)
vocab = sorted(set(text))
print(f"Vocabulary size: {len(vocab)} chars")
stoi  = {ch: i for i, ch in enumerate(vocab)}   # char → int
itos  = {i: ch for ch, i in stoi.items()}        # int  → char

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 90 / 10 split (no shuffling – preserve temporal order)
n_train = int(len(text) * 0.9)
train_text = text[:n_train]
val_text   = text[n_train:]

train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids   = torch.tensor(encode(val_text),   dtype=torch.long)
print(f"Train tokens: {len(train_ids):,}  |  Val tokens: {len(val_ids):,}")


Total characters: 1,115,394
Vocabulary size: 65 chars
Train tokens: 1,003,854  |  Val tokens: 111,540


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Causal Transformer language model                    ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# A decoder-only Transformer: each position can only attend to
# earlier positions (causal mask). Same encoder blocks, but with
# a causal mask passed at each layer.
#
# Architecture:
#   Embedding  : vocab_size → d_model
#   + Sinusoidal PE
#   N × EncoderBlock (with causal mask)
#   LayerNorm
#   Linear head: d_model → vocab_size

VOCAB_SIZE = len(vocab)   # 65

class CausalTransformerLM(nn.Module):
    """Character-level causal Transformer language model.

    Parameters
    ----------
    vocab_size : number of unique characters (65 for Tiny Shakespeare)
    d_model    : embedding dimension
    n_heads    : attention heads
    n_layers   : number of blocks
    context    : maximum context (sequence) length
    dropout    : dropout rate
    """
    def __init__(self,
                 vocab_size: int = VOCAB_SIZE,
                 d_model: int = D_MODEL,
                 n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS,
                 context: int = CONTEXT,
                 dropout: float = DROPOUT):
        super().__init__()
        self.context = context
        self.d_model = d_model

        # Token embedding: (vocab, d)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        # Positional encoding (sinusoidal, fixed)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, context, dropout)

        # Transformer blocks with causal mask
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_model * 4, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        # Output projection → vocabulary logits
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Tie weights: embedding ↔ lm_head (reduces parameters, standard practice)
        self.lm_head.weight = self.tok_emb.weight

        # Build the fixed causal mask once
        # shape (context, context): True = mask out (future)
        causal = torch.triu(torch.ones(context, context, dtype=torch.bool), diagonal=1)
        self.register_buffer('causal_mask', causal)

    def forward(self, idx: torch.Tensor,
                use_mask: bool = True) -> torch.Tensor:
        """Forward pass for training (teacher forcing).

        idx : (B, T) int token indices

        Returns logits : (B, T, vocab_size)
        """
        B, T = idx.shape
        assert T <= self.context

        # Token + positional embeddings
        x = self.tok_emb(idx)       # (B, T, d)
        x = self.pos_enc(x)         # add PE

        # Causal mask (only valid positions)
        mask = self.causal_mask[:T, :T] if use_mask else None

        for block in self.blocks:
            x = block(x, mask=mask)

        x = self.norm(x)
        return self.lm_head(x)      # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx: torch.Tensor,
                 max_new_tokens: int = 500,
                 temperature: float = 1.0,
                 top_k: int = 40) -> torch.Tensor:
        """Autoregressive generation with top-k sampling.

        idx : (1, T) seed context
        Returns extended sequence (1, T + max_new_tokens)
        """
        for _ in range(max_new_tokens):
            # Crop to context window if needed
            idx_cond = idx[:, -self.context:]
            logits   = self(idx_cond)                   # (1, T, vocab)
            logits   = logits[:, -1, :] / temperature   # last step, (1, vocab)

            # Top-k filtering: zero out all but top-k logits
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)            # (1, vocab)
            next_tok = torch.multinomial(probs, 1)       # (1, 1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx

lm = CausalTransformerLM().to(DEVICE)
print(f"Language model parameters: {count_params(lm):,}")
# Quick shape check
dummy_idx = torch.randint(0, VOCAB_SIZE, (2, CONTEXT)).to(DEVICE)
out_lm    = lm(dummy_idx)
assert out_lm.shape == (2, CONTEXT, VOCAB_SIZE)
print(f"LM output shape: {tuple(out_lm.shape)}  ✓")


Language model parameters: 799,616
LM output shape: (2, 128, 65)  ✓


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Train the character-level language model             ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Training strategy:
#   • Randomly sample batches of context-length windows from train_ids
#   • Input  : tokens 0…T-1
#   • Target : tokens 1…T  (next-token prediction)
#   • Loss   : cross-entropy over all positions (average)

BATCH_LM    = 64
N_EPOCHS_LM = 20   # increase to 30+ for better results on T4

def get_batch(data, batch_size=BATCH_LM, context=CONTEXT, device=DEVICE):
    """Sample a random batch of (input, target) token sequences.

    Returns
    -------
    xb : (B, T)   input tokens
    yb : (B, T)   target tokens (shifted by 1)
    """
    ix = torch.randint(len(data) - context, (batch_size,))
    xb = torch.stack([data[i   : i+context  ] for i in ix]).to(device)
    yb = torch.stack([data[i+1 : i+context+1] for i in ix]).to(device)
    return xb, yb

def eval_loss(model, data, n_batches=50):
    """Estimate average cross-entropy loss on a data split."""
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(n_batches):
            xb, yb = get_batch(data)
            logits  = model(xb)                              # (B,T,vocab)
            B, T, V = logits.shape
            loss    = F.cross_entropy(logits.view(B*T, V),
                                      yb.view(B*T))
            losses.append(loss.item())
    return np.mean(losses)

set_seed(0)
lm    = CausalTransformerLM().to(DEVICE)
opt   = torch.optim.AdamW(lm.parameters(), lr=3e-4, weight_decay=0.01)
crit  = nn.CrossEntropyLoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS_LM)

STEPS_PER_EPOCH = 200   # batches per epoch

train_lm_losses, val_lm_losses = [], []
print(f"Training LM ({N_EPOCHS_LM} epochs × {STEPS_PER_EPOCH} steps)…")
t_start = time.time()

for epoch in range(N_EPOCHS_LM):
    lm.train()
    for _ in range(STEPS_PER_EPOCH):
        xb, yb = get_batch(train_ids)
        opt.zero_grad()
        logits = lm(xb)                               # (B,T,vocab)
        B, T, V = logits.shape
        loss   = crit(logits.view(B*T, V), yb.view(B*T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
        opt.step()
    sched.step()

    # Evaluate every epoch
    t_loss = eval_loss(lm, train_ids)
    v_loss = eval_loss(lm, val_ids)
    train_lm_losses.append(t_loss)
    val_lm_losses.append(v_loss)
    bits   = v_loss / math.log(2)
    print(f"  Epoch {epoch+1:3d}/{N_EPOCHS_LM}  "
          f"train={t_loss:.4f}  val={v_loss:.4f}  bits/char={bits:.4f}")

total_lm_time = time.time() - t_start
print(f"\nLM training complete in {total_lm_time/60:.1f} min")
val_bpc = val_lm_losses[-1] / math.log(2)
print(f"Final validation loss: {val_lm_losses[-1]:.4f} nats  ({val_bpc:.4f} bits/char)")


Training LM (20 epochs × 200 steps)…
  Epoch   1/20  train=2.6065  val=2.6290  bits/char=3.7928
  Epoch   2/20  train=2.4929  val=2.5177  bits/char=3.6323
  Epoch   3/20  train=2.4076  val=2.4337  bits/char=3.5110
  Epoch   4/20  train=2.3311  val=2.3566  bits/char=3.3999
  Epoch   5/20  train=2.2738  val=2.3027  bits/char=3.3221
  Epoch   6/20  train=2.2197  val=2.2490  bits/char=3.2446
  Epoch   7/20  train=2.1697  val=2.2138  bits/char=3.1939
  Epoch   8/20  train=2.1356  val=2.1887  bits/char=3.1576
  Epoch   9/20  train=2.0943  val=2.1576  bits/char=3.1127
  Epoch  10/20  train=2.0712  val=2.1443  bits/char=3.0936
  Epoch  11/20  train=2.0435  val=2.1177  bits/char=3.0551
  Epoch  12/20  train=2.0235  val=2.1113  bits/char=3.0459
  Epoch  13/20  train=2.0070  val=2.1056  bits/char=3.0378
  Epoch  14/20  train=1.9919  val=2.0866  bits/char=3.0103
  Epoch  15/20  train=1.9814  val=2.0861  bits/char=3.0096
  Epoch  16/20  train=1.9658  val=2.0748  bits/char=2.9934
  Epoch  17/20  tra

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Figure 9: LM learning curves; Table 4; generation    ║
# ╚══════════════════════════════════════════════════════════════════╝

# Figure 9: learning curves
fig, ax = plt.subplots(figsize=(8, 4))
ep = range(1, len(train_lm_losses)+1)
ax.plot(ep, train_lm_losses, 'tab:blue',  lw=2, label='Train loss')
ax.plot(ep, val_lm_losses,   'tab:orange',lw=2, label='Val loss', ls='--')
ax.axhline(math.log(65), color='grey', ls=':', lw=1.5, label='Uniform baseline ln65≈4.17')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy (nats)')
ax.set_title('Figure 9: Character-level LM learning curves (Tiny Shakespeare)')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/'fig9_lm_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show(); print("Figure 9 saved.")

# Table 4
print("\nTable 4: Language Model – validation metrics")
print(f"  Val loss (nats)  : {val_lm_losses[-1]:.4f}")
print(f"  Bits per char    : {val_lm_losses[-1]/math.log(2):.4f}")
print(f"  Uniform baseline : {math.log(65):.4f} nats  ({1.0:.4f} bits → log2(65)≈6.02 bpc)")

# ── Text generation at 3 temperatures ────────────────────────────────────────
set_seed(42)
seed_text = "ROMEO:\n"
seed_ids  = torch.tensor([encode(seed_text)], dtype=torch.long).to(DEVICE)

generated = {}
for temp in [0.2, 0.8, 1.5]:
    lm.eval()
    out_ids = lm.generate(seed_ids.clone(), max_new_tokens=500,
                          temperature=temp, top_k=40)
    generated[temp] = decode(out_ids[0].tolist())

# Print previews
for temp in [0.2, 0.8, 1.5]:
    print(f"\n── Temperature={temp} ──────────────────────────────────")
    print(generated[temp][:300])
    print("…")

# Save generated_samples.txt
samples_path = '/content/2026AJ05077_DNN_Assignment/problem2/generated_samples.txt'
os.makedirs(os.path.dirname(samples_path), exist_ok=True)   # creates folder if missing
with open(samples_path, 'w') as f:
    for temp in [0.2, 0.8, 1.5]:
        f.write(f"{'='*60}\n")
        f.write(f"Temperature = {temp}\n")
        f.write(f"{'='*60}\n")
        f.write(generated[temp])
        f.write("\n\n")
print("generated_samples.txt saved.")


Figure 9 saved.

Table 4: Language Model – validation metrics
  Val loss (nats)  : 2.0645
  Bits per char    : 2.9785
  Uniform baseline : 4.1744 nats  (1.0000 bits → log2(65)≈6.02 bpc)

── Temperature=0.2 ──────────────────────────────────
ROMEO:
What the not the will the the the sof the the the he the the will hat
The be we the the prear the the the so deare the the the with he be the mare the
The athe the the preath he sof the willl the he and the the the the the the with the he make freath he with willll
The he the the the the the 
…

── Temperature=0.8 ──────────────────────────────────
ROMEO:
O, sef, I dove you dove thou man's dext,
And brost this with sis of spespear art.


QUEEEES ELIZA:
My his as trep whave; and whe bed be was be tot bastt
Thing lut she wall leve me bich ade's may acer all heeads,
May bedor speend whall fand me herear faris ton
s we and enof dit thiss of my the
…

── Temperature=1.5 ──────────────────────────────────
ROMEO:
Wourr hok is new cuper en, the dot i

## Task 5 – Analysis and Ablations

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 16 — Figure 10: Two attention head heatmaps (Task 5a)     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# We pick a 20-character excerpt from Shakespeare, run it through the
# trained LM, and visualise two attention heads from the first block.

sample_sentence = "To be or not to be, t"   # 21 chars
assert len(sample_sentence) <= CONTEXT

lm.eval()
tok_ids = torch.tensor([encode(sample_sentence)], dtype=torch.long).to(DEVICE)
with torch.no_grad():
    _ = lm(tok_ids)

# Retrieve attention weights from block 0
attn_weights = lm.blocks[0].attn.last_weights   # (1, n_heads, T, T)
# attn_weights: (1, 4, 21, 21)

T_disp = len(sample_sentence)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
chars     = list(sample_sentence)

for ax_idx, head_idx in enumerate([0, 1]):
    w    = attn_weights[0, head_idx, :T_disp, :T_disp].cpu().numpy()
    im   = axes[ax_idx].imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    axes[ax_idx].set_xticks(range(T_disp)); axes[ax_idx].set_xticklabels(chars, fontsize=7)
    axes[ax_idx].set_yticks(range(T_disp)); axes[ax_idx].set_yticklabels(chars, fontsize=7)
    axes[ax_idx].set_title(f'Head {head_idx+1} – Block 1', fontsize=10)
    axes[ax_idx].set_xlabel('Key position (attended to)')
    axes[ax_idx].set_ylabel('Query position')
    plt.colorbar(im, ax=axes[ax_idx], shrink=0.8)

fig.suptitle('Figure 10: Attention heatmaps (excerpt: "To be or not to be, t")',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR/'fig10_attention_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 10 saved.")
print("\nInterpretation notes:")
print("  Head 1 tends to show local/previous-token attention patterns.")
print("  Head 2 may show longer-range dependencies or positional patterns.")
print("  Both heads respect the causal mask (zero weight above diagonal).")


Figure 10 saved.

Interpretation notes:
  Head 1 tends to show local/previous-token attention patterns.
  Head 2 may show longer-range dependencies or positional patterns.
  Both heads respect the causal mask (zero weight above diagonal).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 17 — Figure 11: Heads ablation (Task 5b)                  ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Fix d=128, vary n_heads in {1, 4, 8}, train briefly, compare val loss.

N_HEADS_ABLATION = [1, 4, 8]
N_EPOCHS_HEADS   = 10   # short run to compare quickly

heads_results = {}
print("Heads ablation…")
for n_h in N_HEADS_ABLATION:
    set_seed(0)
    model_h = CausalTransformerLM(n_heads=n_h).to(DEVICE)
    opt_h   = torch.optim.AdamW(model_h.parameters(), lr=3e-4, weight_decay=0.01)
    v_losses_h = []
    model_h.train()
    for ep in range(N_EPOCHS_HEADS):
        for _ in range(100):
            xb, yb = get_batch(train_ids)
            opt_h.zero_grad()
            logits = model_h(xb)
            B,T,V  = logits.shape
            F.cross_entropy(logits.view(B*T,V), yb.view(B*T)).backward()
            torch.nn.utils.clip_grad_norm_(model_h.parameters(), 1.0)
            opt_h.step()
        vl = eval_loss(model_h, val_ids, n_batches=30)
        v_losses_h.append(vl)
    heads_results[n_h] = v_losses_h
    print(f"  heads={n_h}: final val loss={v_losses_h[-1]:.4f}")

# Figure 11
fig, ax = plt.subplots(figsize=(7, 4))
for n_h in N_HEADS_ABLATION:
    ax.plot(range(1, N_EPOCHS_HEADS+1), heads_results[n_h],
            marker='o', ms=4, label=f'{n_h} head(s)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation loss (nats)')
ax.set_title('Figure 11: Heads ablation – val loss vs number of heads (d=128)')
ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/'fig11_heads_ablation.png', dpi=150, bbox_inches='tight')
plt.show(); print("Figure 11 saved.")


Heads ablation…
  heads=1: final val loss=2.2042
  heads=4: final val loss=2.2370
  heads=8: final val loss=2.2523
Figure 11 saved.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 18 — Figure 12: No-mask ablation (Task 5c)                ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Without the causal mask the model sees future tokens during training.
# This causes extremely low training loss (cheating) but poor generation,
# since at test time future tokens are not available.

N_EPOCHS_NM = 10
print("No-mask ablation…")

set_seed(0)
lm_no_mask = CausalTransformerLM().to(DEVICE)
opt_nm     = torch.optim.AdamW(lm_no_mask.parameters(), lr=3e-4)
t_losses_nm, v_losses_nm = [], []

for ep in range(N_EPOCHS_NM):
    lm_no_mask.train()
    for _ in range(100):
        xb, yb = get_batch(train_ids)
        opt_nm.zero_grad()
        logits = lm_no_mask(xb, use_mask=False)   # ← no causal mask
        B,T,V  = logits.shape
        loss   = F.cross_entropy(logits.view(B*T,V), yb.view(B*T))
        loss.backward()
        opt_nm.step()
    with torch.no_grad():
        tl = np.mean([F.cross_entropy(
                 lm_no_mask(xb, use_mask=False).view(-1,VOCAB_SIZE),
                 yb.view(-1)).item() for xb,yb in [get_batch(train_ids) for _ in range(20)]])
        vl = eval_loss(lm_no_mask, val_ids, n_batches=20)
    t_losses_nm.append(tl); v_losses_nm.append(vl)
    print(f"  Epoch {ep+1}: train={tl:.4f} val={vl:.4f}")

# Compare masked vs unmasked learning curves
set_seed(0)
lm_masked = CausalTransformerLM().to(DEVICE)
opt_m2    = torch.optim.AdamW(lm_masked.parameters(), lr=3e-4)
t_losses_m, v_losses_m = [], []
for ep in range(N_EPOCHS_NM):
    lm_masked.train()
    for _ in range(100):
        xb, yb = get_batch(train_ids)
        opt_m2.zero_grad()
        logits = lm_masked(xb, use_mask=True)
        B,T,V  = logits.shape
        F.cross_entropy(logits.view(B*T,V), yb.view(B*T)).backward()
        opt_m2.step()
    with torch.no_grad():
        tl = np.mean([F.cross_entropy(
                 lm_masked(xb).view(-1,VOCAB_SIZE),
                 yb.view(-1)).item() for xb,yb in [get_batch(train_ids) for _ in range(20)]])
        vl = eval_loss(lm_masked, val_ids, n_batches=20)
    t_losses_m.append(tl); v_losses_m.append(vl)

# Figure 12
fig, axes = plt.subplots(1, 2, figsize=(12,4), sharey=True)
ep = range(1, N_EPOCHS_NM+1)
axes[0].plot(ep, t_losses_m,  'tab:blue',   lw=2, label='Train (masked)')
axes[0].plot(ep, v_losses_m,  'tab:blue',   lw=2, ls='--', label='Val (masked)')
axes[0].plot(ep, t_losses_nm, 'tab:red',    lw=2, label='Train (no mask)')
axes[0].plot(ep, v_losses_nm, 'tab:red',    lw=2, ls='--', label='Val (no mask)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CE loss (nats)')
axes[0].set_title('Training & val loss'); axes[0].legend(fontsize=8)

# Generation comparison
lm_no_mask.eval()
seed_ids = torch.tensor([encode("ROMEO:\n")], dtype=torch.long).to(DEVICE)
gen_nm = lm_no_mask.generate(seed_ids, max_new_tokens=200, temperature=0.8, top_k=40)
gen_nm_text = decode(gen_nm[0].tolist())

axes[1].axis('off')
axes[1].text(0.05, 0.95, "Generated text (no mask, T=0.8):\n" + gen_nm_text[:300],
             transform=axes[1].transAxes, va='top', fontsize=7,
             wrap=True, fontfamily='monospace')

fig.suptitle('Figure 12: No-mask ablation – training loss collapse and broken generation',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR/'fig12_no_mask_ablation.png', dpi=150, bbox_inches='tight')
plt.show(); print("Figure 12 saved.")


No-mask ablation…
  Epoch 1: train=4.8623 val=4.1010
  Epoch 2: train=3.8084 val=3.0954
  Epoch 3: train=3.4524 val=2.8687
  Epoch 4: train=3.2754 val=2.7784
  Epoch 5: train=3.1299 val=2.6954
  Epoch 6: train=2.9923 val=2.6518
  Epoch 7: train=2.8748 val=2.6241
  Epoch 8: train=2.7982 val=2.6125
  Epoch 9: train=2.7381 val=2.5985
  Epoch 10: train=2.6916 val=2.6014
Figure 12 saved.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 19 — Figure 13: Time & memory vs context length (Task 5d) ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Scaled dot-product attention computes Q K^T of shape (B,h,T,T),
# so memory and FLOPs scale as O(T²·d). We measure wall-clock time
# and GPU memory at context = 64, 128, 256, 512.

CONTEXT_VALUES = [64, 128, 256, 512]
B_PROBE        = 8   # fixed batch size

times_ctx, mems_ctx = [], []

set_seed(0)
probe_model = CausalTransformerLM(context=max(CONTEXT_VALUES)).to(DEVICE)
probe_model.eval()

print("Measuring time and memory vs context length…")
for ctx in CONTEXT_VALUES:
    # Build dummy input
    idx = torch.randint(0, VOCAB_SIZE, (B_PROBE, ctx)).to(DEVICE)

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    # Warm up
    with torch.no_grad():
        _ = probe_model(idx)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

    # Timed run (average of 20 iterations)
    t0 = time.time()
    for _ in range(20):
        with torch.no_grad():
            _ = probe_model(idx)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.time() - t0) / 20 * 1000   # ms

    mem = 0.0
    if DEVICE.type == 'cuda':
        mem = torch.cuda.max_memory_allocated() / 1e6   # MB

    times_ctx.append(elapsed)
    mems_ctx.append(mem)
    print(f"  ctx={ctx:4d}: {elapsed:.2f} ms  |  peak mem={mem:.1f} MB")

# Figure 13
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(CONTEXT_VALUES, times_ctx, 'o-', color='tab:blue', lw=2)
ax1.set_xlabel('Context length T')
ax1.set_ylabel('Time per forward pass (ms)')
ax1.set_title('Forward-pass time vs context length')
ax1.set_xticks(CONTEXT_VALUES)
# Reference O(T²) curve
T2_ref = [times_ctx[0] * (c / CONTEXT_VALUES[0])**2 for c in CONTEXT_VALUES]
ax1.plot(CONTEXT_VALUES, T2_ref, 'k--', lw=1, label='O(T²) reference')
ax1.legend()

ax2.plot(CONTEXT_VALUES, mems_ctx, 's-', color='tab:red', lw=2)
ax2.set_xlabel('Context length T')
ax2.set_ylabel('Peak GPU memory (MB)')
ax2.set_title('Memory vs context length')
ax2.set_xticks(CONTEXT_VALUES)
if max(mems_ctx) > 0:
    mem2_ref = [mems_ctx[0] * (c / CONTEXT_VALUES[0])**2 for c in CONTEXT_VALUES]
    ax2.plot(CONTEXT_VALUES, mem2_ref, 'k--', lw=1, label='O(T²) reference')
    ax2.legend()

fig.suptitle('Figure 13: Time and memory vs context length (O(T²·d) scaling)',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR/'fig13_time_memory_context.png', dpi=150, bbox_inches='tight')
plt.show(); print("Figure 13 saved.")


Measuring time and memory vs context length…
  ctx=  64: 3.73 ms  |  peak mem=377.9 MB
  ctx= 128: 4.25 ms  |  peak mem=389.2 MB
  ctx= 256: 3.65 ms  |  peak mem=430.6 MB
  ctx= 512: 10.30 ms  |  peak mem=589.0 MB
Figure 13 saved.


## Save results.csv (Problem 2 rows)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 20 — Merge P1 + P2 results into results.csv              ║
# ╚══════════════════════════════════════════════════════════════════╝

rows_p2 = []

# Task A: Transformer HAR (3 seeds, sinusoidal)
for s_idx, seed in enumerate(SEEDS):
    r = sin_multi[s_idx]
    rows_p2.append({
        'problem': 2, 'model':'Transformer(sinusoidal)', 'seed':seed,
        'seq_len':128, 'test_acc':round(r['test_acc'],4),
        'macro_f1':round(r['macro_f1'],4), 'test_mse':'',
        'params':r['params'], 'sec_per_epoch':round(r['sec_per_epoch'],3),
        'val_loss_nats':'', 'bits_per_char':'', 'notes':'HAR classification Task A',
    })

# Positional encoding ablation (seed 0)
for pt in ['sinusoidal','learned','none']:
    r = pos_results[pt]
    rows_p2.append({
        'problem':2,'model':f'Transformer({pt})','seed':0,
        'seq_len':128,'test_acc':round(r['test_acc'],4),
        'macro_f1':round(r['macro_f1'],4),'test_mse':'',
        'params':r['params'],'sec_per_epoch':round(r['sec_per_epoch'],3),
        'val_loss_nats':'','bits_per_char':'','notes':'Task A pos_enc ablation',
    })

# Task B: LM
bpc = val_lm_losses[-1] / math.log(2)
rows_p2.append({
    'problem':2,'model':'CausalTransformerLM','seed':0,
    'seq_len':CONTEXT,'test_acc':'','macro_f1':'','test_mse':'',
    'params':count_params(lm),
    'sec_per_epoch':round(total_lm_time/N_EPOCHS_LM,2),
    'val_loss_nats':round(val_lm_losses[-1],4),
    'bits_per_char':round(bpc,4),
    'notes':'Task B character LM',
})

# Heads ablation
for n_h in N_HEADS_ABLATION:
    rows_p2.append({
        'problem':2,'model':f'Transformer(heads={n_h})','seed':0,
        'seq_len':CONTEXT,'test_acc':'','macro_f1':'','test_mse':'',
        'params':'','sec_per_epoch':'',
        'val_loss_nats':round(heads_results[n_h][-1],4),
        'bits_per_char':round(heads_results[n_h][-1]/math.log(2),4),
        'notes':'Task 5b heads ablation',
    })

df_p2 = pd.DataFrame(rows_p2, columns=[
    'problem','model','seed','seq_len','test_acc','macro_f1',
    'test_mse','params','sec_per_epoch','val_loss_nats','bits_per_char','notes'])

# Try to merge with P1 results (may not exist if running P2 standalone)
p1_csv = '../results_p1.csv'
if os.path.isfile(p1_csv):
    df_p1 = pd.read_csv(p1_csv)
    df_all = pd.concat([df_p1, df_p2], ignore_index=True)
else:
    df_all = df_p2
    print("Note: results_p1.csv not found — saving P2 rows only.")

df_all.to_csv('../../results.csv', index=False)
print(f"results.csv saved  ({len(df_all)} rows)")


Note: results_p1.csv not found — saving P2 rows only.
results.csv saved  (10 rows)
